In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max, udf
from pyspark.sql.types import LongType
import struct

# Initialize Spark session with the necessary configuration
spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "LEGACY") \
    .getOrCreate()

# Load data to the dataframe
df = spark.read.table("DE_LH_BRONZE_ELC.Item_Ledger_Entry")

# Define a UDF to convert binary to bigint
def binary_to_bigint(binary_value):
    return struct.unpack('>q', binary_value)[0]

binary_to_bigint_udf = udf(binary_to_bigint, LongType())

# Apply the UDF to create a new column with the bigint representation of the timestamp
df = df.withColumn("timestamp_bigint", binary_to_bigint_udf(col("timestamp")))

# Find the maximum value in the timestamp_bigint column
max_timestamp_row = df.select(max(col("timestamp_bigint")).alias("max_sequence")).collect()[0]
max_timestamp_sequence = max_timestamp_row['max_sequence']

# Get the corresponding [Entry_No_] value
max_timestamp_no_row = df.filter(col("timestamp_bigint") == max_timestamp_sequence).select("Entry_No_").collect()[0]
max_timestamp_no = max_timestamp_no_row['Entry_No_']

# Print the maximum timestamp sequence number and corresponding [Entry_No_] value
print(f"Max timestamp sequence number: {max_timestamp_sequence}")
print(f"Corresponding [Entry_No_] value: {max_timestamp_no}")

# Set the whereClause parameter
mssparkutils.notebook.exit(f"{max_timestamp_sequence}")

StatementMeta(, 670835ba-b129-407b-950d-cf86a051b0cc, 3, Finished, Available, Finished)

Max timestamp sequence number: 15316948626
Corresponding [Entry_No_] value: 17114335
ExitValue: 15316948626